## Executive Summary

**Purpose:** Everything the gear shop and the inspector need for a helical pair: flank tolerances, tooth thickness allowances that give the intended backlash, the gear data table for each drawing, and a transverse outline for CAD.

**What it does:** `helical_tolerances` gives the ISO 1328-1:2013 tolerances. `pair_backlash` sets the normal tooth thickness allowances of pinion and gear and reports the resulting normal backlash range. `helical_data_sheet` writes the ISO 1340 / DIN 3966-1 data table, with helix angle, hand, lead, base helix, span over $k$ teeth in the normal section, and dimension over balls. `helical_outline` gives the transverse tooth outline at mean thickness, which `gears.manufacturing.write_dxf` writes as a DXF.

**Units:** mm and µm (tolerances).

## How It Fits Into the Helical Gear Workflow

**Upstream:** `gears.helical.geometry` and `gears.helical.layout`, `gears.manufacturing` (ISO 286 limits, ISO 1328 formulas, allowance type, data row type, case depth, DXF writer, tooth outline), `gears.rating` (materials).

**Downstream:** `gears.helical.design` (package files).

In [ ]:
#| default_exp gears.helical.manufacturing

In [ ]:
#| export
from __future__ import annotations

import math
from types import SimpleNamespace

import numpy as np

from emachines.gears.manufacturing import (iso1328_tolerances, it_grade, iso286_limits, ThicknessAllowance, DataRow,
                                           case_depth, gear_outline, _round_up)
from emachines.gears.rating import GearMaterial
from emachines.gears.helical.geometry import (HelicalGear, helical_span_teeth, helical_span_measurement, span_fits,
                                              helical_ball_diameter, helical_ball_measurement)
from emachines.gears.helical.layout import HelicalPair

## Flank Tolerances (ISO 1328-1:2013)

ISO 1328-1:2013 writes every tolerance as a function of the reference diameter $d$, the **normal** module $m_n$ and the face width $b$, scaled by $\sqrt2^{\,A-5}$ for flank tolerance class $A$. These are the same formulas as for spur gears (`gears.manufacturing.iso1328_tolerances`). The only difference is that a helical gear's reference diameter is $z\,m_n/\cos\beta$. The helix tolerances $F_{\beta T}$ and $f_{H\beta T}$ matter more for a helical gear, because the overlap spreads the load along the helix.

In [ ]:
#| export
def helical_tolerances(g: HelicalGear, A: int) -> dict[str, float]:
    "ISO 1328-1:2013 flank tolerances [µm] of a helical gear for flank tolerance class A."
    return iso1328_tolerances(SimpleNamespace(d=g.d, m=g.mn, b=g.b), A)  # type: ignore[arg-type]

In [ ]:
#| hide
from emachines.gears.involute import GearGeometry
_g = HelicalGear(20, 2.0, 0.0, b=20)
assert helical_tolerances(_g, 6) == iso1328_tolerances(GearGeometry(20, 2.0, b=20), 6)
assert helical_tolerances(HelicalGear(20, 2.0, 0.3, b=20), 6)["F_pT"] >= helical_tolerances(_g, 6)["F_pT"]

## Backlash and Tooth Thickness

The pair needs a minimum normal backlash $j_{bn,min}$ for lubrication, thermal growth and the tolerances it must absorb. The value follows the practice of `gears.manufacturing` (AGMA 2002 style):

$$j_{bn,min} = \tfrac23\,(0.06 + 0.0005\,a_w + 0.03\,m_n) .$$

A thinner tooth adds normal backlash $-A_{sn}\cos\alpha_n$, measured along the line of action in the normal plane. A smaller centre distance, by $f_a$, takes away $2f_a\sin\alpha_{wt}\cos\beta_b$. With the housing tolerance $\pm f_a$ (IT7/2), the sum of the upper allowances must be

$$-(A_{sne1} + A_{sne2}) = \frac{j_{bn,min} + 2f_a\sin\alpha_{wt}\cos\beta_b}{\cos\alpha_n},$$

and it is split equally between pinion and gear, rounded up to 5 µm. The thickness tolerance of each gear covers its radial runout, $T_{sn} = 2F_r\tan\alpha_n + 15$ µm, as for the spur gears. The backlash range is then

$$j_{bn} = -(A_{sn1} + A_{sn2})\cos\alpha_n \mp 2f_a\sin\alpha_{wt}\cos\beta_b .$$

In [ ]:
#| export
def pair_backlash(pair: HelicalPair, A: int) -> dict:
    "Normal thickness allowances of pinion and gear and the normal backlash range [mm] (see above)."
    p, g, mc = pair.pinion, pair.gear, pair.mesh
    a_w = pair.a_w
    f_a = it_grade(a_w, 7) / 2
    jmin = 2 / 3 * (0.06 + 0.0005 * a_w + 0.03 * p.mn)
    ca = math.cos(p.alpha_n)
    k_a = 2 * f_a * math.sin(mc.alpha_wt) * math.cos(p.beta_b)
    S = (jmin + k_a) / ca
    up = _round_up(S / 2)
    out = {}
    for key, gg in (("pinion", p), ("gear", g)):
        Fr = helical_tolerances(gg, A)["F_rT"] / 1000
        T = _round_up(2 * Fr * math.tan(gg.alpha_n) + 0.015)
        out[key] = ThicknessAllowance(-up, -up - T)
    j_lo = -(out["pinion"].A_sne + out["gear"].A_sne) * ca - k_a
    j_hi = -(out["pinion"].A_sni + out["gear"].A_sni) * ca + k_a
    return {"allowances": out, "backlash_n": (j_lo, j_hi), "j_bn_min_required": jmin, "f_a": f_a,
            "backlash_t": tuple(j / (math.cos(p.beta_b) * math.cos(mc.alpha_wt)) for j in (j_lo, j_hi))}

The last entry, `backlash_t`, is the circumferential backlash on the pitch circle, $j_t = j_{bn}/(\cos\alpha_{wt}\cos\beta_b)$. It is what a dial indicator shows when it rocks the gear against a held pinion, which is how a robot joint's lost motion is usually specified.

In [ ]:
import math
pair = HelicalPair.build(17, 52, mn=1.5, beta=math.radians(15), b=20)
bl = pair_backlash(pair, 6)
print(bl["allowances"], "j_bn = {:.3f} … {:.3f} mm".format(*bl["backlash_n"]))

In [ ]:
#| hide
assert bl["backlash_n"][0] >= bl["j_bn_min_required"] - 1e-9
assert bl["backlash_n"][1] > bl["backlash_n"][0]
assert all(a.A_sne < 0 and a.A_sni < a.A_sne for a in bl["allowances"].values())

## Gear Data Table

The data table on each gear drawing follows ISO 1340 / DIN 3966-1. Beyond the spur entries it gives the helix angle and **hand**, the transverse module and pressure angle, the base helix angle and the **lead**, which the gear shop needs to set up the hob and to check the helix. The thickness is given as the **normal** tooth thickness with its allowances. It is converted to:

- the **span** $W_k$ over $k$ teeth in the normal plane, when the face is wide enough for the micrometer discs ($b \ge W_k\sin\beta_b + 2$ mm, `span_fits`), and
- the **dimension over two balls** $M_d$, always. The table says whether the tooth count is odd, because then the two balls sit across one tooth space in the same transverse plane.

The tolerances are the ISO 1328-1 values of the chosen class. The helix slope tolerance carries a crowning note for the pinion. A light crown of 3–5 µm (or a helix angle correction) makes up for shaft bending and housing misalignment.

In [ ]:
#| export
def _dev(d: float, fit: str) -> str:
    up, lo = iso286_limits(d, fit)
    f = lambda v: "0" if abs(v) < 5e-7 else f"{v:+.3f}"
    return f"({f(up)}/{f(lo)})"


def _dms(a: float) -> str:
    "Degrees as d° m.mm′."
    tm = round(a * 60, 2)
    d = int(tm // 60)
    return f"{d}° {tm - 60 * d:.2f}′"


def _route(g: HelicalGear, mat: GearMaterial, finish: str, integral: bool) -> str:
    blank = "turn shaft between centres (journals as datum)" if integral else "turn blank (bore and faces as datum)"
    steps = [blank, "hob (helical, hob swivelled to β)"]
    if mat.kind == "case":
        steps += ["carburize, harden, temper",
                  "grind journals and shoulders" if integral else "grind bore and faces",
                  "grind flanks (generating or profile grinding)" if finish == "ground" else "hone flanks"]
    elif mat.kind == "nitrided":
        steps += ["shave or grind before nitriding" if finish == "ground" else "", "gas nitride"]
    elif finish == "ground":
        steps += ["grind flanks"]
    return " → ".join(s for s in steps if s)


def helical_data_sheet(g: HelicalGear, mate: HelicalGear, a_w: float, allowance: ThicknessAllowance, A: int,
                       mat: GearMaterial, finish: str, d_active: float | None = None,
                       integral: bool = False) -> list[DataRow]:
    """
    ISO 1340 / DIN 3966-1 data table rows of helical gear g meshing with `mate` at centre distance a_w.

    finish: 'ground' or 'hobbed'; d_active: start of active profile (from `helical_mesh_check`);
    integral: the teeth are cut on a shaft (process route).
    """
    deg = math.degrees
    s_up, s_lo = g.sn + allowance.A_sne, g.sn + allowance.A_sni
    tol = helical_tolerances(g, A)
    rows = [
        DataRow("Gear", "", g.name),
        DataRow("Type", "", "helical, external" if g.beta > 0 else "spur, external"),
        DataRow("Number of teeth", "z", f"{g.z}"),
        DataRow("Normal module", "m_n", f"{g.mn:g} mm"),
        DataRow("Normal pressure angle", "α_n", f"{deg(g.alpha_n):g}°"),
        DataRow("Helix angle", "β", f"{deg(g.beta):.4f}° ({_dms(deg(g.beta))})"),
        DataRow("Hand of helix", "", g.hand if g.beta > 0 else "–"),
        DataRow("Transverse module", "m_t", f"{g.mt:.5f} mm"),
        DataRow("Transverse pressure angle", "α_t", f"{deg(g.alpha_t):.4f}°"),
        DataRow("Base helix angle", "β_b", f"{deg(g.beta_b):.4f}°"),
        DataRow("Lead", "p_z", "–" if g.beta == 0 else f"{g.lead:.3f} mm"),
        DataRow("Basic rack profile", "", f"ISO 53 profile A: h_aP = {g.ha:g} m_n, h_fP = {g.hf:g} m_n, ρ_fP = {g.rho_f:g} m_n"),
        DataRow("Profile shift coefficient", "x_n", f"{g.x:+.4f}"),
        DataRow("Reference diameter", "d", f"{g.d:.4f} mm"),
        DataRow("Base diameter", "d_b", f"{g.db:.4f} mm"),
        DataRow("Tip diameter", "d_a", f"{g.da:.3f} mm h11 " + _dev(g.da, "h11")),
        DataRow("Root diameter", "d_f", f"{g.df:.3f} mm (max)"),
        DataRow("Root form diameter", "d_Ff", f"≤ {g.d_form:.3f} mm" + (f" (active profile starts at {d_active:.3f} mm)" if d_active else "")),
        DataRow("Face width", "b", f"{g.b:g} mm ±0.1"),
        DataRow("Normal tooth thickness on d (arc)", "s_n", f"{g.sn:.4f} mm, allowances {allowance.A_sne:+.3f}/{allowance.A_sni:+.3f} mm"),
    ]
    k = helical_span_teeth(g)
    Wu, Wl = helical_span_measurement(g, s_up, k), helical_span_measurement(g, s_lo, k)
    fits = span_fits(g, k)
    rows.append(DataRow("Span measurement (normal plane)", f"W_k (k = {k})", f"{Wu:.4f} / {Wl:.4f} mm"
                        + ("" if fits else f" — face too narrow for discs (needs b ≥ {Wu * math.sin(g.beta_b) + 2:.1f} mm); use balls")))
    D = helical_ball_diameter(g)
    Mu, _ = helical_ball_measurement(g, D, s_up)
    Ml, _ = helical_ball_measurement(g, D, s_lo)
    rows.append(DataRow("Dimension over balls", f"M_d (D_M = {D:.2f} mm)", f"{Mu:.4f} / {Ml:.4f} mm"
                        + (" (odd z: balls in one transverse plane, across one tooth space)" if g.z % 2 else "")))
    rows += [
        DataRow("Flank tolerance class", "", f"ISO 1328-1:2013 class {A}"),
        DataRow("Single pitch tolerance", "f_pT", f"{tol['f_pT']:g} µm"),
        DataRow("Total cumulative pitch tolerance", "F_pT", f"{tol['F_pT']:g} µm"),
        DataRow("Total profile tolerance", "F_αT", f"{tol['F_alphaT']:g} µm"),
        DataRow("Profile slope tolerance", "f_HαT", f"±{tol['f_HalphaT']:g} µm"),
        DataRow("Total helix tolerance", "F_βT", f"{tol['F_betaT']:g} µm"),
        DataRow("Helix slope tolerance", "f_HβT", f"±{tol['f_HbetaT']:g} µm" + (" (crown 3–5 µm recommended)" if g.name == "pinion" else "")),
        DataRow("Radial runout tolerance", "F_rT", f"{tol['F_rT']:g} µm"),
        DataRow("Mating gear", "", f"{mate.name}, z = {mate.z}, {mate.hand} hand"),
        DataRow("Centre distance", "a_w", f"{a_w:.4f} mm ±{it_grade(a_w, 7) / 2 * 1000:.0f} µm"),
        DataRow("Material", "", mat.name),
        DataRow("Heat treatment", "", mat.treatment),
        DataRow("Hardness", "", mat.hardness),
        DataRow("Hardening depth", "", case_depth(g.virtual(), mat)),
        DataRow("Flank roughness", "", {"ground": "Ra ≤ 0.4 µm (Rz ≤ 3)", "hobbed": "Ra ≤ 1.6 µm"}[finish]),
        DataRow("Root roughness", "", "Ra ≤ 3.2 µm, no grinding notch in the root"),
        DataRow("Tip edge", "", f"chamfer or round 0.1–0.15 m_n ({0.1 * g.mn:.2f}–{0.15 * g.mn:.2f} mm); "
                                "break the acute end-face edge of the helix"),
        DataRow("Process route", "", _route(g, mat, finish, integral)),
    ]
    return rows

In [ ]:
from emachines.gears.rating import MATERIALS
for row in helical_data_sheet(pair.pinion, pair.gear, pair.a_w, bl["allowances"]["pinion"], 6,
                              MATERIALS["case_carburized"], "ground", pair.mesh.d_Nf1, integral=True)[:24]:
    print(f"{row.item:36s} {row.symbol:22s} {row.value}")

In [ ]:
#| hide
_rows = {r.item: r.value for r in helical_data_sheet(pair.pinion, pair.gear, pair.a_w, bl["allowances"]["pinion"], 6,
                                                     MATERIALS["case_carburized"], "ground")}
assert _rows["Hand of helix"] == "right" and "left hand" in _rows["Mating gear"]
assert _rows["Lead"].endswith("mm") and "15° 0.00′" in _rows["Helix angle"]
_W = float(_rows["Span measurement (normal plane)"].split(" / ")[0])
assert math.isclose(_W, helical_span_measurement(pair.pinion, pair.pinion.sn + bl["allowances"]["pinion"].A_sne), abs_tol=1e-4)

## Tooth Outline for CAD

The DXF holds one transverse section, the end face of the gear. That is what a CAD model extrudes with a twist of $360°\cdot b/p_z$ (left or right by hand) to make the helical gear. The outline is the involute of the transverse equivalent from `gears.manufacturing.gear_outline`, at the mean of the thickness allowances. A normal thickness change $\Delta s_n$ is $\Delta s_n/\cos\beta$ in the transverse section. The root fillet is the transverse rack's circle of the radius chosen in `gears.helical.geometry`. That is close to, but not exactly, the ellipse a hob makes, which is fine for layout and assembly, and for a solid model to check fits.

In [ ]:
#| export
def helical_outline(g: HelicalGear, ds_n: float | None = None) -> np.ndarray:
    "Closed transverse outline (N×2, mm) of helical gear g, normal thickness change ds_n [mm] (default 0)."
    ds_t = 0.0 if ds_n is None else ds_n / math.cos(g.beta)
    return gear_outline(g.transverse(), ds_t)


def twist_angle(g: HelicalGear) -> float:
    "Twist of the transverse section over the face width [°], positive for a right-hand helix (CAD sweep)."
    if g.beta == 0:
        return 0.0
    t = 360.0 * g.b / g.lead
    return t if g.hand == "right" else -t

In [ ]:
#| hide
_P = helical_outline(pair.pinion, -0.03)
_r = np.hypot(_P[:, 0], _P[:, 1])
assert abs(_r.max() - pair.pinion.da / 2) < 1e-6 and _r.min() >= pair.pinion.df / 2 - 1e-3
assert twist_angle(pair.pinion) > 0 > twist_angle(pair.gear)
assert math.isclose(abs(twist_angle(pair.pinion)), 360 * 20 / pair.pinion.lead)

In [ ]:
#| echo: false
import matplotlib.pyplot as plt

def _pair_figure():
    fig, ax = plt.subplots(figsize=(8, 4.5))
    P1 = helical_outline(pair.pinion)
    th = np.pi + np.pi / pair.gear.z
    R = np.array([[np.cos(th), np.sin(th)], [-np.sin(th), np.cos(th)]])
    P2 = helical_outline(pair.gear) @ R + [pair.a_w, 0]
    ax.fill(*P1.T, fc="#d8e8d0", ec="#222", lw=0.6)
    ax.fill(*P2.T, fc="#f6e7c8", ec="#222", lw=0.6)
    for c_, r_ in ((0, pair.d_w1 / 2), (pair.a_w, pair.d_w2 / 2)):
        ax.add_patch(plt.Circle((c_, 0), r_, fill=False, ls="--", ec="#1f5fa8", lw=0.6))
    ax.set_aspect("equal"); ax.set_xlim(-18, 50); ax.set_ylim(-17, 17); ax.axis("off")
    ax.set_title("17/52 pair, m_n 1.5, β 15°: transverse section with operating pitch circles", fontsize=10)
    plt.show()
_pair_figure()

## References

1. ISO 1328-1:2013, *Cylindrical gears — ISO system of flank tolerance classification — Part 1*.
2. ISO 1340:1976, *Cylindrical gears — Information to be given to the manufacturer by the purchaser*; DIN 3966-1:1978, *Information on gear teeth in drawings*.
3. DIN 3967:1978, *System of gear fits — Backlash, tooth thickness allowances, tooth thickness tolerances*.
4. ISO 21771:2007 (span and ball measurements of helical gears).
5. AGMA 2002-B88, *Tooth Thickness Specification and Measurement* (minimum backlash).